**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/masset-2025"], check=True)
    os.chdir("/content/encore/entries/masset-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Masset et al. (2025): VR task, single-mouse fits (DANDI:000251)

**Paper.** Masset P, Tano P, Kim HR, Malik AN, Pouget A, Uchida N (2025). Multi-timescale
reinforcement learning in the brain. *Nature* 642:682-690. doi:10.1038/s41586-025-08929-9.
The claims file (`claims/masset-2025.yaml`) was written from the accepted author manuscript in
PubMed Central (main text, legends, Methods and Extended Data legends), with the paper's Mendeley
code (doi:10.17632/tc43t3s7c5.1) and simulation code (github.com/pablotano8/multi_timescale_RL at
37f11c6).

**Data.** DANDI:000251, version `draft`, modified 2023-06-20T00:56:22Z. The dandiset has no
published version, so the draft is the only version there is; it is pinned by its modification
time, which the notebook checks. It holds the data of Kim et al. (2020), which the paper reuses:
the VR task single units, one unit per file, and no cued delay task recordings. The cued delay
units are in the paper's own deposit, Mendeley Data doi:10.17632/tc43t3s7c5.1 (version 1,
2025-04-09), file `DA_SU_ODOR_NEW_W_OL.h5`; its `FinalAverageArray.mat` (the paper's per-neuron
table) and `Valid_S.mat` (the paper's neuron flags) link the two tasks' units. All files are
streamed into memory.

**Seeds.** Random draws use each claim's seed, `zlib.crc32(b"masset-2025/<claim id>")`, and four
more (plus 1000 to 4000); a second generator of a claim adds 1. Shared computations use the seed
of the claim that defines them: the VR half splits that of `fig3b-vr-ramp-population`, the VR fit's
initial values that of `fig3-common-value-fit`, the cued delay fits that of `cued-delay-neurons`.

The entry has four notebooks: `vr_population.ipynb` (VR claims on the population),
`vr_single_mouse.ipynb` (single-mouse VR fits), `vr_reward_expectation.ipynb` (the common reward
expectation model), `cued_delay_and_cross_task.ipynb` (link, cued delay and cross-task claims). The claims each one tests are listed below.

**This notebook.** Tested: ed8ej-single-animal-vr-fits. It refits the population at the claims'
seeds (as `vr_population.ipynb` does) to compare with.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
import time
import zlib
import collections
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import multiscale as ms

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
T0 = time.time()
OFFSETS = [0, 1000, 2000, 3000, 4000]


def seed(claim, offset=0):
    return zlib.crc32(f"masset-2025/{claim}".encode()) + offset


# --- load
info = ms.dandi_version_info()
print("dandiset", ms.DANDISET, "version", ms.DANDI_VERSION, "modified", info["modified"])
assert info["modified"].startswith(ms.DANDI_MODIFIED[:19])
assets = ms.dandi_assets()
files = ms.dandi_su_files(assets, kinds=("SU-VR-OL",))
print(f"loaded in {time.time() - T0:.0f} s")

## Data check

The open-loop units and the M6 selection, as in `vr_population.ipynb`, where the files are
checked in full.

In [ ]:
vr, vr_dropped = ms.dandi_vr_units(files)
vr_rates = [ms.vr_trial_rates(u) for u in vr]
vr_mean = np.array([r.mean() for r in vr_rates])
vr_ml = np.array([u["ml"] for u in vr])
vr_mouse = np.array([u["mouse"] for u in vr])
vr_sel = (vr_mean > ms.RATE_MIN) & (vr_ml < ms.ML_MAX)
rates_sel = [r for r, s in zip(vr_rates, vr_sel) if s]
n_vr = len(rates_sel)
print(f"open-loop units {len(vr)} (left out: {vr_dropped}); selected (M6): {n_vr}")
print("by mouse:", dict(sorted(collections.Counter(vr_mouse[vr_sel]).items())))


def vr_run(off):
    ps = ms.vr_bootstrap_psths(rates_sel, np.random.default_rng(seed("fig3b-vr-ramp-population", off)))
    x0 = ms.cv_initial(n_vr, np.random.default_rng(seed("fig3-common-value-fit", off)))
    return dict(ps=ps, slopes=ms.ramp_slopes(ps), fits=ms.cv_fit_bootstraps(ps, x0))


t = time.time()
VR = {0: vr_run(0)}
f = VR[0]["fits"]
print(f"population fit at the claims' seeds (as in vr_population.ipynb): {time.time() - t:.0f} s; "
      f"gamma mean {f['gamma'].mean(0).mean():.3f}, s.d. {f['gamma'].mean(0).std(ddof=1):.3f}")

mice_sel = vr_mouse[vr_sel]
for m in (3044, 3054):
    k = vr_mouse == m
    print(f"mouse {m}: {k.sum()} open-loop units, {(k & vr_sel).sum()} selected; sessions "
          f"{len(set(u['start'] for u, kk in zip(vr, k) if kk))}; ML {sorted(set(vr_ml[k]))}")

### ed8ej-single-animal-vr-fits

> Fitted to the neurons of one mouse alone (m3044, 29 neurons; m3054, 24 neurons), the common
> value model gives gains, baselines and discount factors strongly correlated with those of the
> full population fit (discount factor r = 0.75 in m3044 and 0.88 in m3054).

The mouse's M6 neurons are fitted alone, with new random splits and initial values (the code's
per-mouse scripts redo both), from the claim's seed (splits) and the seed plus 1 (initial values).
The comparison is with the population fit at the claims' seeds. The code compares unlike gains:
the population script stores a / gamma^0.1 (Master_Script_SingleValue_All_gain.m lines 429 to
431), the per-mouse scripts the fitted a (Master_Script_SingleValue_m3044.m lines 435 to 437), and
VR_Figures_Revision.m (lines 909 and 1051) correlates the two. That is the test (the criterion
cites those lines); the like-for-like gains (a against a, and a / gamma^0.1 against a / gamma^0.1)
are shown as variants.

In [ ]:
def mouse_run(off):
    out = {}
    rng = np.random.default_rng(seed("ed8ej-single-animal-vr-fits", off))
    rng_init = np.random.default_rng(seed("ed8ej-single-animal-vr-fits", off) + 1)
    for mouse in (3044, 3054):
        k = mice_sel == mouse
        ps = ms.vr_bootstrap_psths([r for r, kk in zip(rates_sel, k) if kk], rng)
        f = ms.cv_fit_bootstraps(ps, ms.cv_initial(k.sum(), rng_init))
        full = VR[0]["fits"]
        res = dict(n=int(k.sum()))
        for name, key_m, key_f in (("gain", "a", "gain"), ("baseline", "base", "base"), ("gamma", "gamma", "gamma"),
                                   ("gain a-a", "a", "a"), ("gain a/g-a/g", "gain", "gain")):
            res[name] = stats.spearmanr(f[key_m].mean(0), full[key_f].mean(0)[k])
        out[mouse] = res
    return out

t = time.time()
MR = {off: mouse_run(off) for off in OFFSETS}
print(f"per-mouse fits under 5 seeds: {time.time() - t:.0f} s")
PN = ("gain", "baseline", "gamma", "gain a-a", "gain a/g-a/g")
rows = [dict(seed=off, mouse=m, n=v["n"], **{f"r_{p}": v[p][0] for p in PN},
             **{f"p_{p}": v[p][1] for p in PN}) for off in OFFSETS for m, v in MR[off].items()]
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("r_gain is the code's comparison (population a / gamma^0.1 against per-mouse a); the last two r are the like-for-like variants")
print("expected r: m3044 gain 0.78, baseline 0.75, gamma 0.75; m3054 gain 0.97, baseline 0.98, gamma 0.88 "
      "(each +- 10%); neurons 29 +- 2.9 and 24 +- 2.4")

**Result: ed8ej-single-animal-vr-fits.** Fitted alone, m3044 has 31 neurons (29 +- 2.9) and m3054 23 (24 +- 2.4). Under the claim's seed, with the code's comparison, m3044 gives Spearman r = 0.700 for the gain, 0.796 for the baseline and 0.775 for the discount factor, and m3054 0.964, 0.997 and 0.857, all with p < 1e-4. Five of the six r are within tolerance; the m3044 gain (0.700) is just below 0.702. The like-for-like gains give 0.698 (a against a) and 0.674 (a / gamma^0.1 on both sides) in m3044, also below, and 0.968 and 0.966 in m3054: the gain reading does not change the outcome. Partially reproduced under the criterion. Under the other four seeds the m3044 gain is 0.703 to 0.759 with the code's comparison, within tolerance, which would make the claim reproduced.


## Summary

| Claim | Outcome (claim's seed) |
|---|---|
| ed8ej-single-animal-vr-fits | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

The fits differ from the code's as in `vr_population.ipynb` (optimiser and iteration cap). The
per-mouse fits draw their own splits and initial values, as the code's per-mouse scripts do, and
are compared with the population fit at the claims' seeds.